# MLflow Model Deployment: House Price Prediction

This notebook shows the full life of an ML model using MLflow:

1. **Track** experiments (parameters, metrics, models)
2. **Compare** runs and pick the best model
3. **Register** the best model in the MLflow Model Registry
4. **Deploy** it as a REST API with `mlflow models serve`
5. **Predict** by sending a request to the API

Dataset: California Housing (20,640 districts, 1990 census). Target is `median_house_value` in USD.

## 1. Problem Statement
A real estate team wants to estimate house prices quickly. Building a model in a notebook is easy. The hard part is what comes after:

- Many experiments get run, and nobody remembers which settings gave the best result.
- The final model lives on one laptop as a `.pkl` file, so other apps cannot use it.
- When a new model is ready, there is no clean way to swap it in or roll back.

**Goal:** use MLflow to track every experiment, keep one approved version of the model in a registry, and serve that version as an API any app can call.

In [3]:
# Import required libraries
import os
import sys
import json
import time
import subprocess
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import requests

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)  # hide info logs

print("MLflow version:", mlflow.__version__)

MLflow version: 3.16.0


## 2. Set up MLflow
We use a local SQLite database as the **tracking store**. This also turns on the **Model Registry**.
To see the dashboard, run this in a separate Command Prompt from the project folder:
```
python -m mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```
then open http://127.0.0.1:5000

In [4]:
# Configure MLflow tracking and model registry
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("house-price-prediction")
client = MlflowClient()

MODEL_NAME = "house-price-model"

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", "house-price-prediction")
print("Registered Model Name:", MODEL_NAME)

Tracking URI: sqlite:///mlflow.db
Experiment: house-price-prediction
Registered Model Name: house-price-model


## 3. Load and prepare the data

In [5]:
URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

(20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [6]:
TARGET = "median_house_value"
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = ["ocean_proximity"]

# Preprocessing lives INSIDE the model pipeline, so the deployed API accepts raw data
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (16512, 9)  Test: (4128, 9)


## 4. Track experiments
Each model is trained inside `mlflow.start_run()`. MLflow saves:
- **Params**: model settings
- **Metrics**: RMSE, MAE, R²
- **Model**: the full pipeline, with a **signature** (expected input columns and types) and an input example

In [7]:
candidates = {
    "LinearRegression": LinearRegression(),
    "RandomForest":     RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.1, random_state=42),
}

model_uris = {}  # run_id -> logged model URI
for name, model in candidates.items():
    with mlflow.start_run(run_name=name):
        pipe = Pipeline([("prep", prep), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

        rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        mae  = float(mean_absolute_error(y_test, pred))
        r2   = float(r2_score(y_test, pred))

        mlflow.log_param("model_type", name)
        mlflow.log_params({k: v for k, v in model.get_params().items()
                           if k in ["n_estimators", "max_depth", "learning_rate"]})
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

        signature = infer_signature(X_train, pipe.predict(X_train.head(5)))
        info = mlflow.sklearn.log_model(pipe, name="model", signature=signature,
                                 input_example=X_train.head(3),
                                 serialization_format="cloudpickle")
        model_uris[mlflow.active_run().info.run_id] = info.model_uri
        print(f"{name:18s} RMSE={rmse:,.0f}  MAE={mae:,.0f}  R2={r2:.3f}")

LinearRegression   RMSE=70,059  MAE=50,670  R2=0.625
RandomForest       RMSE=49,379  MAE=32,187  R2=0.814
GradientBoosting   RMSE=47,388  MAE=31,379  R2=0.829


## 5. Compare runs and pick the best model (lowest RMSE)

In [8]:
# Compare the three models trained in this experiment
model_results = []

for run_id, model_uri in model_uris.items():
    run = client.get_run(run_id)
    model_results.append({
        "run_id": run_id,
        "model_type": run.data.params["model_type"],
        "RMSE": run.data.metrics["rmse"],
        "MAE": run.data.metrics["mae"],
        "R2 Score": run.data.metrics["r2"],
        "model_uri": model_uri,
    })

comparison = pd.DataFrame(model_results).sort_values(by="RMSE").reset_index(drop=True)
display(comparison[["model_type", "RMSE", "MAE", "R2 Score"]].round(3))

best_model = comparison.iloc[0]          # lowest RMSE
best_run_id = best_model["run_id"]
best_model_uri = best_model["model_uri"]
print("Best model:", best_model["model_type"])

,model_type,RMSE,MAE,R2 Score
0,GradientBoosting,47387.812,31379.134,0.829
1,RandomForest,49378.695,32187.066,0.814
2,LinearRegression,70059.193,50670.489,0.625


Best model: GradientBoosting


## 6. Register the best model
The **Model Registry** keeps numbered versions of a model. We give the approved version the alias **`champion`**.
Apps always ask for `@champion`, so swapping or rolling back a model only means moving the alias. No code changes needed.

In [9]:
# Register the best-performing model in MLflow
# Note: every time this cell runs, a NEW version is created
registered_model = mlflow.register_model(
    model_uri=best_model_uri,
    name=MODEL_NAME
)

# Point the "champion" alias at this new version
client.set_registered_model_alias(MODEL_NAME, "champion", registered_model.version)

print("MODEL REGISTRATION SUCCESSFUL")
print("-----------------------------")
print("Model Name:", MODEL_NAME)
print("Model Version:", registered_model.version)
print("Model Alias: champion")
print("Model URI:", f"models:/{MODEL_NAME}@champion")

MODEL REGISTRATION SUCCESSFUL
-----------------------------
Model Name: house-price-model
Model Version: 2
Model Alias: champion
Model URI: models:/house-price-model@champion


Registered model 'house-price-model' already exists. Creating a new version of this model...
Created version '2' of model 'house-price-model'.


In [10]:
# Verify registered model and champion alias

model_version = client.get_model_version_by_alias(
    "house-price-model",
    "champion"
)

print("MODEL REGISTRY VERIFICATION")
print("---------------------------")

print("Model Name:", model_version.name)
print("Model Version:", model_version.version)
print("Model Alias:", model_version.aliases)

print("\nModel URI:")
print(f"models:/house-price-model@champion")

MODEL REGISTRY VERIFICATION
---------------------------
Model Name: house-price-model
Model Version: 2
Model Alias: ['champion']

Model URI:
models:/house-price-model@champion


## 7. Load the registered model and predict

In [11]:
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
sample = X_test.head(5)
out = sample[["median_income", "housing_median_age", "ocean_proximity"]].copy()
out["actual"] = y_test.head(5).values
out["predicted"] = champion.predict(sample).round(0)
out

,median_income,housing_median_age,ocean_proximity,actual,predicted
20046,1.6812,25.0,INLAND,47700.0,47318.0
3024,2.5313,30.0,INLAND,45800.0,71338.0
15663,3.4801,52.0,NEAR BAY,500001.0,438588.0
20484,5.7376,17.0,<1H OCEAN,218600.0,252088.0
9814,3.7250,34.0,NEAR OCEAN,278000.0,274788.0


## 8. Deploy as a REST API
`mlflow models serve` wraps the registered model in a web server. Any app (website, Excel, mobile) can now send house details and get a price back.

In a Windows Command Prompt you would run:
```
set MLFLOW_TRACKING_URI=sqlite:///mlflow.db
python -m mlflow models serve -m "models:/house-price-model@champion" -p 5001 --env-manager local
```
Here we start it from Python so the whole demo runs in one notebook.

In [12]:
# Start MLflow Model Serving
API_URL = "http://127.0.0.1:5001"

try:
    # If a server is already running on port 5001, reuse it
    requests.get(f"{API_URL}/ping", timeout=3)
    print("MLflow server is already running at", API_URL)

except requests.exceptions.RequestException:
    env = os.environ.copy()
    env["MLFLOW_TRACKING_URI"] = mlflow.get_tracking_uri()

    # Put the .venv Scripts folder first, so Windows uses the .venv uvicorn
    venv_scripts = str(Path(sys.executable).parent)
    env["PATH"] = venv_scripts + os.pathsep + env["PATH"]

    log_file = open("model_server.log", "w", encoding="utf-8")
    server = subprocess.Popen(
        [sys.executable, "-m", "mlflow", "models", "serve",
         "-m", f"models:/{MODEL_NAME}@champion",
         "--host", "127.0.0.1", "--port", "5001",
         "--env-manager", "local"],
        env=env, stdout=log_file, stderr=subprocess.STDOUT)

    # Wait until the server answers (up to 90 seconds)
    for _ in range(90):
        if server.poll() is not None:
            print("Server crashed. Log below:\n")
            log_file.close()
            print(Path("model_server.log").read_text(encoding="utf-8", errors="replace")[-3000:])
            break
        try:
            if requests.get(f"{API_URL}/ping", timeout=2).status_code == 200:
                print("Model server is up at", API_URL)
                break
        except requests.exceptions.RequestException:
            time.sleep(1)

Model server is up at http://127.0.0.1:5001


## 9. Call the API with a new house

In [13]:
# Check MLflow REST API health
API_URL = "http://127.0.0.1:5001"

try:
    response = requests.get(f"{API_URL}/ping", timeout=5)

    print("API HEALTH CHECK")
    print("----------------")
    print("Status Code:", response.status_code)
    print("Response:", response.text)

    if response.status_code == 200:
        print("API is ready to receive predictions.")

except requests.exceptions.RequestException as e:
    print("API health check failed:", e)


API HEALTH CHECK
----------------
Status Code: 200
Response: 

API is ready to receive predictions.


In [14]:
# Send new house details to the MLflow REST API

new_houses = pd.DataFrame([
    {
        "longitude": -122.25,
        "latitude": 37.85,
        "housing_median_age": 30,
        "total_rooms": 2000,
        "total_bedrooms": 400,
        "population": 900,
        "households": 380,
        "median_income": 7.5,
        "ocean_proximity": "NEAR BAY"
    },
    {
        "longitude": -118.30,
        "latitude": 34.05,
        "housing_median_age": 40,
        "total_rooms": 1500,
        "total_bedrooms": 350,
        "population": 1200,
        "households": 330,
        "median_income": 2.5,
        "ocean_proximity": "<1H OCEAN"
    }
])

# Prepare JSON payload
payload = {
    "dataframe_split": new_houses.to_dict(orient="split")
}

# Send prediction request
response = requests.post(
    "http://127.0.0.1:5001/invocations",
    json=payload,
    timeout=30
)

print("API STATUS:", response.status_code)

if response.status_code == 200:

    predictions = response.json()["predictions"]

    results = new_houses.copy()
    results["predicted_house_price"] = [
        round(float(p), 2) for p in predictions
    ]

    print("\nHOUSE PRICE PREDICTIONS")
    display(results)

else:
    print("PREDICTION ERROR:")
    print(response.text)


API STATUS: 200

HOUSE PRICE PREDICTIONS


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,predicted_house_price
0,-122.25,37.85,30,2000,400,900,380,7.5,NEAR BAY,369957.02
1,-118.30,34.05,40,1500,350,1200,330,2.5,<1H OCEAN,188487.79


The same request with `curl` (what another app would send):

In [13]:
print("curl -X POST http://127.0.0.1:5001/invocations -H 'Content-Type: application/json' -d '"
      + json.dumps({"dataframe_split": new_houses.head(1).to_dict(orient="split")}) + "'")

curl -X POST http://127.0.0.1:5001/invocations -H 'Content-Type: application/json' -d '{"dataframe_split": {"index": [0], "columns": ["longitude", "latitude", "housing_median_age", "total_rooms", "total_bedrooms", "population", "households", "median_income", "ocean_proximity"], "data": [[-122.25, 37.85, 30, 2000, 400, 900, 380, 7.5, "NEAR BAY"]]}}'


In [14]:
# Stop MLflow Model Serving
if "server" in globals() and server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=10)
        print("MLflow server stopped successfully.")
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
        print("MLflow server was forcefully stopped.")
else:
    print("No active server process found.")

if "log_file" in globals() and not log_file.closed:
    log_file.close()

MLflow server stopped successfully.


## 10. Learnings
- **Tracking removes guesswork.** Every run's settings, scores and model file are saved, so results are easy to compare and repeat.
- **Put preprocessing inside the pipeline.** The API then takes raw house details, and training and serving always match.
- **Signatures catch bad input.** MLflow checks column names and types before the model runs.
- **Aliases make deployment safe.** Apps call `@champion`. Promoting a new model or rolling back is a one line change in the registry.
- **One command to serve.** `mlflow models serve` turns any logged model into a REST API without writing Flask code.
- **Next steps:** run the tracking server remotely, package with `mlflow models build-docker`, and deploy the container to a cloud service.